In [1]:
# ============================================================
# Phase 3: Germinal handoff -> Hugging Face ESM2 metric scoring
# Saves reranked outputs to Google Drive
# ============================================================

# 0. Setup
from google.colab import drive
drive.mount("/content/drive")

!pip install -q transformers huggingface_hub accelerate pandas tqdm biopython

import os, re, glob, json
import torch
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForSequenceClassification

PROJECT_DIR = "/content/drive/MyDrive/generative-ai-project"
HANDOFF_DIR = f"{PROJECT_DIR}/results/germinal_phase2/handoffs"
OUT_DIR = f"{PROJECT_DIR}/results/germinal_phase3/esm2_scores"
os.makedirs(OUT_DIR, exist_ok=True)

INPUT_CSV = f"{HANDOFF_DIR}/pdl1_smoke_all_candidates.csv"

HF_REPO = "zhangtin/antibody-esm2-finetuned"
BASE_MODEL = "facebook/esm2_t33_650M_UR50D"   # checkpoint hidden dim = 1280

METRICS = {
    "binding_koenig": "koenig_binding_g6/fold0_best.pt",
    "expression_koenig": "koenig_expression_g6/fold0_best.pt",
    "binding_warszawski": "warszawski_binding_d44/fold0_best.pt",
    # add your developability checkpoints here if uploaded:
    # "sec_monomer": "gdp/SEC_monomer/fold0_best.pt",
    # "hic_retention": "gdp/HIC_retention/fold0_best.pt",
    # "acsins_ph6": "gdp/AC_SINS_pH6/fold0_best.pt",
    # "tm1": "gdp/Tm1/fold0_best.pt",
}

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.8 MB/s eta 0:00:00


In [3]:
# 1. Load Germinal candidates

df = pd.read_csv(INPUT_CSV)
print("Loaded:", df.shape)
print(df.columns.tolist())

def pick_sequence_col(df):
    for c in ["sequence", "trajectory_sequence", "binder_sequence", "seq"]:
        if c in df.columns:
            return c
    raise ValueError("No sequence column found.")

SEQ_COL = pick_sequence_col(df)
print("Using sequence column:", SEQ_COL)

df.head()

Loaded: (8, 96)
['candidate_id', 'generator', 'target', 'run_id', 'seed', 'chain_format', 'sequence', 'structure_backend', 'status', 'output_structure_path', 'abmpnn_score', 'abmpnn_seqid', 'alpha_all', 'alpha_interface', 'beta_all', 'beta_interface', 'beta_strand', 'binder_near_hotspot', 'binder_near_hotspot_filter', 'binder_score', 'cdr3_hotspot_contacts', 'cdr3_hotspot_contacts_filter', 'cdr_hotspot_contacts', 'cdr_lengths', 'cdr_sap', 'clashes', 'clashes_filter', 'clashes_unrelaxed', 'design_name', 'design_time', 'experiment_name', 'external_aggregate_score', 'external_binder_pae', 'external_chain_ptm', 'external_i_pae', 'external_i_plddt', 'external_iptm', 'external_iptm_filter', 'external_pae', 'external_pae_filter', 'external_plddt', 'external_plddt_binder', 'external_plddt_filter', 'external_ptm', 'external_ptm_filter', 'final_structure_path', 'framework_mutations', 'helix', 'hydrophobic_patches_binder', 'hydrophobic_patches_struct', 'i_pae', 'i_ptm', 'interface_AA', 'interface

,candidate_id,generator,target,run_id,seed,chain_format,sequence,structure_backend,status,output_structure_path,...,sap_score,sc_rmsd,sc_rmsd_filter,ss_content,surface_hydrophobicity,surface_hydrophobicity_filter,target_hotspots,target_len,trajectory_pdb_af,trajectory_sequence
0,pdl1_scfv_s979559_abmpnn_2,germinal,pdl1,phase2_pdl1_smoke,979559,scfv,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKLYICWVRQAPGKGLE...,chai,redesign_candidate,/home/zichende/projects/generative-ai-project-...,...,82.4260,3.32,True,"{'alpha_': 2.48, 'beta_': 49.59, 'loops_': 47....",0.36,True,"A37,A39,A41,A96,A98",115,/home/zichende/projects/generative-ai-project-...,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKLYICWVRQAPGKGLE...
1,pdl1_scfv_s979559_abmpnn_4,germinal,pdl1,phase2_pdl1_smoke,979559,scfv,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKLYICWVRQAPGKGLE...,chai,redesign_candidate,/home/zichende/projects/generative-ai-project-...,...,80.8963,3.91,True,"{'alpha_': 3.72, 'beta_': 48.76, 'loops_': 47....",0.34,True,"A37,A39,A41,A96,A98",115,/home/zichende/projects/generative-ai-project-...,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKLYICWVRQAPGKGLE...
2,pdl1_scfv_s494416_abmpnn_3,germinal,pdl1,phase2_pdl1_smoke,494416,scfv,EVQLVESGGGLVQPGGSLRLSCAASQSLGSNTVIHWVRQAPGKGLE...,chai,redesign_candidate,/home/zichende/projects/generative-ai-project-...,...,76.4448,9.07,False,"{'alpha_': 3.72, 'beta_': 47.11, 'loops_': 49....",0.35,True,"A37,A39,A41,A96,A98",115,/home/zichende/projects/generative-ai-project-...,EVQLVESGGGLVQPGGSLRLSCAASQSLGSNTVIHWVRQAPGKGLE...
3,pdl1_scfv_s979559_abmpnn_3,germinal,pdl1,phase2_pdl1_smoke,979559,scfv,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKDYICWVRQAPGKGLE...,chai,redesign_candidate,/home/zichende/projects/generative-ai-project-...,...,89.9140,5.53,True,"{'alpha_': 4.96, 'beta_': 49.59, 'loops_': 45....",0.39,True,"A37,A39,A41,A96,A98",115,/home/zichende/projects/generative-ai-project-...,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKDYICWVRQAPGKGLE...
4,pdl1_scfv_s979559_abmpnn_1,germinal,pdl1,phase2_pdl1_smoke,979559,scfv,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKTYICWVRQAPGKGLE...,chai,redesign_candidate,/home/zichende/projects/generative-ai-project-...,...,81.2374,8.69,False,"{'alpha_': 4.96, 'beta_': 50.83, 'loops_': 44....",0.39,True,"A37,A39,A41,A96,A98",115,/home/zichende/projects/generative-ai-project-...,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKTYICWVRQAPGKGLE...


In [4]:
# 2. Clean / QC sequences

AA = set("ACDEFGHIKLMNPQRSTVWY")

def clean_seq(s):
    s = str(s).upper()
    s = re.sub(r"[^A-Z]", "", s)
    return s

df["phase3_sequence"] = df[SEQ_COL].map(clean_seq)

before = len(df)
df = df[df["phase3_sequence"].str.len().between(30, 1022)].copy()
df = df[df["phase3_sequence"].map(lambda x: set(x).issubset(AA))].copy()
df = df.drop_duplicates(subset=["phase3_sequence"]).copy()

print(f"QC kept {len(df)} / {before} candidates")

QC kept 8 / 8 candidates


In [5]:
# 3. Model loader for your Hugging Face checkpoints

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def load_metric_model(metric_name, ckpt_file):
    ckpt_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=ckpt_file,
        repo_type="model"
    )

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt

    # Your checkpoint uses esm2.*; HF sequence classifier expects esm.*
    remapped = {}
    for k, v in state.items():
        nk = k
        if nk.startswith("esm2."):
            nk = nk.replace("esm2.", "esm.", 1)
        remapped[nk] = v

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL,
        num_labels=1,
        problem_type="regression"
    )

    missing, unexpected = model.load_state_dict(remapped, strict=False)
    model.to(device).eval()

    print(f"\n[{metric_name}] loaded")
    print("missing head/base keys:", len(missing))
    print("unexpected keys:", len(unexpected))
    if isinstance(ckpt, dict) and "val_metrics" in ckpt:
        print("val_metrics:", {k: v for k, v in ckpt["val_metrics"].items() if k in ["spearman","pearson","rmse","mae","r2"]})

    return model

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [6]:
# 4. Batched scoring

@torch.no_grad()
def score_sequences(model, seqs, batch_size=4):
    scores = []
    for i in tqdm(range(0, len(seqs), batch_size)):
        batch = seqs[i:i+batch_size]
        toks = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=1022,
            return_tensors="pt"
        ).to(device)

        out = model(**toks)
        scores.extend(out.logits.squeeze(-1).detach().cpu().float().numpy().tolist())

    return np.array(scores, dtype=float)

In [7]:
# 5. Score all requested metrics

seqs = df["phase3_sequence"].tolist()

for metric_name, ckpt_file in METRICS.items():
    model = load_metric_model(metric_name, ckpt_file)
    df[f"esm2_{metric_name}"] = score_sequences(model, seqs, batch_size=4)
    del model
    torch.cuda.empty_cache()

score_cols = [c for c in df.columns if c.startswith("esm2_")]
df[score_cols].describe()

koenig_binding_g6/fold0_best.pt:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[binding_koenig] loaded
missing head/base keys: 37
unexpected keys: 9
val_metrics: {'spearman': np.float64(0.4909166351152071), 'pearson': np.float32(0.48171264), 'rmse': 0.4936921719596724, 'mae': 0.3360826075077057, 'r2': 0.22746145725250244}


...

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[binding_warszawski] loaded
missing head/base keys: 37
unexpected keys: 9
val_metrics: {'spearman': np.float64(0.6019287411889391), 'pearson': np.float32(0.59255093), 'rmse': 1.0540799687032516, 'mae': 0.8293677568435669, 'r2': 0.3351668119430542}


  0%|          | 0/2 [00:00<?, ?it/s]

,esm2_binding_koenig,esm2_expression_koenig,esm2_binding_warszawski
count,8.000000,8.000000,8.000000
mean,-0.037590,0.053303,0.115798
std,0.003100,0.007059,0.008993
min,-0.042463,0.041194,0.101036
25%,-0.039152,0.053129,0.112412
50%,-0.037750,0.056854,0.119325
75%,-0.035687,0.057018,0.121454
max,-0.032959,0.058136,0.124751


In [8]:
# 6. Composite ranking
# Higher is assumed better for binding/expression.
# For future liability metrics where lower is better, invert them below.

higher_better = [
    "esm2_binding_koenig",
    "esm2_expression_koenig",
    "esm2_binding_warszawski",
]

lower_better = [
    # "esm2_hic_retention",
    # "esm2_acsins_ph6",
]

def zscore(x):
    return (x - x.mean()) / (x.std() + 1e-8)

for c in higher_better:
    if c in df.columns:
        df[c + "_z"] = zscore(df[c])

for c in lower_better:
    if c in df.columns:
        df[c + "_z"] = -zscore(df[c])

zcols = [c for c in df.columns if c.endswith("_z")]
df["phase3_composite_score"] = df[zcols].mean(axis=1)

df = df.sort_values("phase3_composite_score", ascending=False).reset_index(drop=True)
df["phase3_rank"] = np.arange(1, len(df) + 1)

In [9]:
# 7. Save reranked results to Google Drive

target_name = "pdl1"
out_csv = f"{OUT_DIR}/{target_name}_phase3_esm2_reranked_candidates.csv"
top_csv = f"{OUT_DIR}/{target_name}_phase3_top50_candidates.csv"

df.to_csv(out_csv, index=False)
df.head(50).to_csv(top_csv, index=False)

print("Saved:")
print(out_csv)
print(top_csv)

df[["phase3_rank", "candidate_id", "phase3_composite_score"] + score_cols + ["phase3_sequence"]].head(20)

Saved:
/content/drive/MyDrive/generative-ai-project/results/germinal_phase3/esm2_scores/pdl1_phase3_esm2_reranked_candidates.csv
/content/drive/MyDrive/generative-ai-project/results/germinal_phase3/esm2_scores/pdl1_phase3_top50_candidates.csv


,phase3_rank,candidate_id,phase3_composite_score,esm2_binding_koenig,esm2_expression_koenig,esm2_binding_warszawski,phase3_sequence
0,1,pdl1_scfv_s979559_abmpnn_3,0.831314,-0.034555,0.056970,0.124751,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKDYICWVRQAPGKGLE...
1,2,pdl1_scfv_s494416_abmpnn_2,0.327262,-0.038438,0.058136,0.120928,EVQLVESGGGLVQPGGSLRLSCAASQSLSSNTVIHWVRQAPGKGLE...
2,3,pdl1_scfv_s979559_abmpnn_1,0.323744,-0.036064,0.056848,0.115588,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKTYICWVRQAPGKGLE...
3,4,pdl1_scfv_s979559_abmpnn_2,0.201671,-0.032959,0.057164,0.102884,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKLYICWVRQAPGKGLE...
4,5,pdl1_scfv_s494416_abmpnn_3,0.107723,-0.038714,0.056632,0.117721,EVQLVESGGGLVQPGGSLRLSCAASQSLGSNTVIHWVRQAPGKGLE...
5,6,pdl1_scfv_s979559_abmpnn_4,-0.322474,-0.037063,0.056860,0.101036,EVQLVESGGGLVQPGGSLRLSCAASQKLSSKLYICWVRQAPGKGLE...
6,7,pdl1_scfv_s494416_abmpnn_4,-0.682105,-0.040467,0.041194,0.121169,EVQLVESGGGLVQPGGSLRLSCAASQSLSSNTVIHWVRQAPGKGLE...
7,8,pdl1_scfv_s494416_abmpnn_1,-0.787135,-0.042463,0.042621,0.122308,EVQLVESGGGLVQPGGSLRLSCAASQSLSSNTVIHWVRQAPGKGLE...


In [10]:
# 8. Optional: save a compact handoff for Phase 4 structure validation

phase4_cols = [
    "phase3_rank",
    "candidate_id",
    "target",
    "run_id",
    "phase3_sequence",
    "phase3_composite_score",
] + score_cols

phase4_cols = [c for c in phase4_cols if c in df.columns]

phase4_csv = f"{OUT_DIR}/{target_name}_phase4_structure_validation_handoff.csv"
df.loc[df["phase3_rank"] <= 50, phase4_cols].to_csv(phase4_csv, index=False)

print("Phase 4 handoff saved:")
print(phase4_csv)

Phase 4 handoff saved:
/content/drive/MyDrive/generative-ai-project/results/germinal_phase3/esm2_scores/pdl1_phase4_structure_validation_handoff.csv
